# MCP Chapter 3 — MCP Primitives

Chapter 2 covered JSON-RPC, initialization, capability discovery, and transports.

This chapter covers the three core MCP primitives:

- **Tools** → actions the model can perform
- **Resources** → data/context the application can read
- **Prompts** → reusable instructions/messages

This notebook uses the **MCP Python SDK v2**. In v2, the old `FastMCP` server API has been replaced by `MCPServer`.

## 1. The three primitives

| Primitive | Purpose | Main idea |
|---|---|---|
| Tool | Perform an action | **Do something** |
| Resource | Provide data/context | **Give me something** |
| Prompt | Create reusable instructions | **Tell the model how to approach something** |

A single MCP server can expose all three.

## 2. Tools

A **Tool** is a function that the model can call to perform an action.

Examples:

- `search_products(query)`
- `get_weather(city)`
- `delete_file(path)`
- `calculate_tax(amount)`

The SDK can use the function's **type hints** to describe its inputs and the docstring to describe the tool.

In [ ]:
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("Chapter 3 Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

From this function, MCP can expose:

- tool name → `add`
- description → `Add two numbers`
- `a` → integer input
- `b` → integer input

The important idea is that a normal Python function becomes an MCP-exposed capability.

In [ ]:
users = {
    101: {"name": "Alice", "role": "admin"},
    102: {"name": "Bob", "role": "developer"},
    103: {"name": "Charlie", "role": "viewer"},
}

@mcp.tool()
def get_user(user_id: int) -> dict:
    """Return information about a user by ID."""
    return users.get(user_id, {"error": "User not found"})

The **model/host decides when to call a tool**. The MCP server executes the function and returns the result.

## 3. Resources

A **Resource** exposes data that an application can read and provide as context.

Examples:

- configuration
- documentation
- files
- database records
- application state

Resources are addressed using **URIs**, such as:

```text
config://app
user://101
file://reports/report.txt
```

The key distinction:

> **Tool = action. Resource = data/context.**

In [ ]:
@mcp.resource("config://app")
def get_config() -> str:
    """Return the application configuration."""
    return "environment=development\nversion=1.0"

The client addresses this resource as:

```text
config://app
```

The function's return value becomes the resource content.

## 4. Resource Templates

A resource template contains a variable in its URI.

Instead of creating separate resources for:

```text
user://101
user://102
user://103
```

one template can handle all users.

In [ ]:
@mcp.resource("user://{user_id}")
def user_resource(user_id: int) -> str:
    """Return information about a user."""
    user = users.get(user_id)

    if user is None:
        return "User not found"

    return f"Name: {user['name']}\nRole: {user['role']}"

Here `{user_id}` is a URI placeholder.

- `config://app` → fixed resource
- `user://{user_id}` → resource template
- The placeholder name should match the function parameter.

## 5. Prompts

A **Prompt** is a reusable prompt template exposed by the server.

It can accept arguments and produce instructions/messages that a host can give to a model.

Examples:

- summarize a document
- review code
- analyze a table
- explain an error

A Prompt is not primarily an action like a Tool.

In [ ]:
@mcp.prompt()
def summarize_text(text: str, style: str = "short") -> str:
    """Create a reusable summarization prompt."""
    if style == "detailed":
        instruction = "Give a detailed summary with the important points."
    else:
        instruction = "Give a short summary with only the key points."

    return f"{instruction}\n\nText:\n{text}"

Think of the three primitives like this:

```text
Tool
Model → tool → action/result

Resource
Application → resource → data/context

Prompt
User/application → prompt → prepared instructions
```

## 6. One server can expose all three

For a company database, one MCP server could expose:

- `search_customer()` → **Tool**
- `customer://{id}` → **Resource**
- `analyze_customer()` → **Prompt**

All three are capabilities of the same MCP server.

In [ ]:
@mcp.tool()
def search_customer(name: str) -> list[dict]:
    """Search customers by name."""
    return [
        user for user in users.values()
        if name.lower() in user["name"].lower()
    ]

@mcp.resource("customer://{user_id}")
def customer_resource(user_id: int) -> str:
    """Return customer information."""
    user = users.get(user_id)

    if user is None:
        return "Customer not found"

    return f"Customer: {user['name']}\nRole: {user['role']}"

@mcp.prompt()
def customer_analysis(customer_name: str) -> str:
    """Create a prompt for analyzing a customer."""
    return (
        f"Analyze the customer '{customer_name}'. "
        "Identify useful observations and potential follow-up actions."
    )

## 7. Discovery

MCP clients can discover these capabilities through the protocol:

```text
Client
  ├── tools/list
  ├── resources/list
  └── prompts/list
```

The server returns metadata describing what is available.

For tools, this includes information such as the name, description, and input schema.

For resources, the client learns available resource addresses/templates.

For prompts, the client learns prompt names and arguments.

## 8. Practical comparison

| Example | Primitive | Why |
|---|---|---|
| `delete_user(101)` | Tool | Changes something |
| `search_users("Alice")` | Tool | Performs an operation |
| `user://101` | Resource | Provides information |
| `config://app` | Resource | Provides application data |
| `review_code(code)` | Prompt | Creates reusable instructions |

A useful mental model:

**Tool = Action**  
**Resource = Context**  
**Prompt = Instructions**

## 9. MCP SDK v2

Older tutorials may use:

```python
from mcp.server.fastmcp import FastMCP
```

In the SDK version used for this chapter, use:

```python
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("Demo")
```

The decorators are:

```python
@mcp.tool()
@mcp.resource(...)
@mcp.prompt()
```

So the MCP concepts remain the same, but the server API/import path changed.

## 10. Mini practical exercise

Build a small task-manager MCP server.

It should expose:

1. A **tool** `add_task(title)` that adds a task.
2. A **tool** `complete_task(task_id)` that marks a task complete.
3. A **resource template** `task://{task_id}` that returns task information.
4. A **prompt** `review_tasks()` that asks an LLM to review the current tasks.

Do not connect a real MCP client yet. Focus on correctly defining the primitives.

## 11. Chapter 3 recap

- MCP has three core primitives: **Tools, Resources, Prompts**.
- A **Tool** lets the model perform an action.
- A **Resource** exposes data/context through a URI.
- A **Resource Template** uses URI placeholders such as `user://{user_id}`.
- A **Prompt** provides reusable instructions/messages.
- Tool input schemas can be derived from Python type hints.
- One MCP server can expose all three primitives.
- MCP SDK v2 uses `MCPServer` rather than the old `FastMCP` import.

Answer these in your own words.



1. What is the main difference between a Tool and a Resource?



2. Why does a Resource use a URI?



3. What is the purpose of a Resource Template?



4. What information can the MCP SDK derive from a tool's Python function?



5. What problem do Prompts solve?



6. If `delete_user(user_id)` changes a database, which MCP primitive should it be? Why?



7. If `user://101` provides information about user 101, which primitive should it be? Why?



8. Can one MCP server expose Tools, Resources, and Prompts at the same time?



9. What changed from `FastMCP` to `MCPServer` in MCP SDK v2?



10. In one sentence each, explain: **Tool = ?, Resource = ?, Prompt = ?**